# GEPA on a larger adversarial AI-text benchmark

This isolated experiment asks a narrow teaching question: **can a reflective prompt optimizer recover a useful classifier when the unoptimized task model is near chance?** It preserves the original 74 Chapter 6 rows, expands the benchmark to 300 rows from traceable real text, and holds out 80 rows before GEPA sees any data.

This is a pedagogical optimizer stress test, not evidence that AI-text detection works reliably in the wild. The holdout was made adversarial using baseline-only predictions, so the result should not be generalized beyond this benchmark.

In [ ]:
from pathlib import Path
import json
import os
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'chapter06' else Path.cwd()
RESULTS = ROOT / 'chapter06' / 'results' / 'gepa_expanded'
DATA = ROOT / 'data' / 'ai_vs_human_chapter06_expanded.csv'
SPLITS = ROOT / 'data' / 'ai_vs_human_chapter06_expanded_splits.json'
FINAL = RESULTS / 'final'
RUN_PAID_GEPA = os.getenv('RUN_PAID_GEPA') == '1'
print(f'Paid execution enabled: {RUN_PAID_GEPA}')

## Dataset design

Each semantic pair contains one real human passage and one Sol rewrite. Pairs—not individual rows—are assigned to train, validation, or test, preventing near-duplicate leakage. Human passages come from pinned technical documentation and December 2019 Wikipedia revisions; every row carries its URL, title, author or publisher, and license.

In [ ]:
dataset = pd.read_csv(DATA)
split_manifest = json.loads(SPLITS.read_text())
source_family = dataset['source_id'].str.startswith('wikipedia-').map({True: 'Wikipedia', False: 'Technical documentation'})
pd.DataFrame({
    'measure': ['Rows', 'Semantic pairs', 'Original rows preserved', 'Sources', 'Train rows', 'Validation rows', 'Locked test rows'],
    'value': [len(dataset), dataset.pair_id.nunique(), 74, dataset.source_id.nunique(),
              2 * len(split_manifest['splits']['train']),
              2 * len(split_manifest['splits']['validation']),
              2 * len(split_manifest['splits']['test'])],
})

In [ ]:
pd.DataFrame({
    'family': source_family,
    'source_id': dataset.source_id,
}).drop_duplicates().groupby('family').agg(sources=('source_id', 'nunique'))

## Baseline gate

Luna is stochastic, so both baseline and optimized programs are measured with three fresh, uncached runs. The preregistered score is the per-example majority prediction. We retain the full run range rather than hiding it.

In [ ]:
baseline = json.loads((RESULTS / 'baseline_confirmation' / 'summary.json').read_text())
pd.DataFrame({
    'reference': ['Run 1', 'Run 2', 'Run 3', 'Per-example majority'],
    'accuracy_pct': baseline['accuracies_pct'] + [baseline['baseline_test_accuracy_pct']],
})

## Optional paid smoke test

The notebook is artifact-first: opening or running it performs no paid calls. Set `RUN_PAID_GEPA=1` only when intentionally reproducing the experiment. The smoke compile uses 12 training rows, 8 validation rows, one full-evaluation budget, and **zero test rows**.

In [ ]:
if RUN_PAID_GEPA:
    from chapter06.experiments.gepa_expanded.run_experiment import run_gepa
    smoke = run_gepa(
        mode='smoke', output_dir=RESULTS / 'gepa_smoke',
        max_full_evals=1, stage_cap_usd=2.0, threads=4,
    )
    display(smoke)
else:
    print('Skipped. Set RUN_PAID_GEPA=1 to reproduce paid optimizer calls.')

## Validation-only GEPA selection

The full compile uses all 160 training rows and all 60 validation rows. The recorded experiment first ran six full evaluations, then seeded an eight-evaluation current-best refinement from the strongest frozen prompt. Candidate selection and refinement use validation only. The locked test remains unavailable until a frozen candidate clears the validation release gate.

In [ ]:
if RUN_PAID_GEPA:
    from chapter06.experiments.gepa_expanded.run_experiment import evaluate_repeated, run_gepa
    reproduction = RESULTS / 'reproduction'
    first_dir = reproduction / 'gepa_candidate_1'
    first = run_gepa(
        mode='full', output_dir=first_dir,
        max_full_evals=6, stage_cap_usd=10.0, threads=4,
    )
    first_validation = evaluate_repeated(
        program_path=first_dir / 'optimized_program.json',
        split_name='validation', output_dir=first_dir / 'validation',
        repeats=3, stage_cap_usd=2.0, threads=4,
    )
    candidate_dir = reproduction / 'gepa_candidate_2'
    candidate = run_gepa(
        mode='full', output_dir=candidate_dir, max_full_evals=8,
        stage_cap_usd=12.0, threads=4,
        seed_program_path=first_dir / 'optimized_program.json',
        candidate_selection_strategy='current_best',
    )
    validation = evaluate_repeated(
        program_path=candidate_dir / 'optimized_program.json',
        split_name='validation', output_dir=candidate_dir / 'validation',
        repeats=3, stage_cap_usd=2.0, threads=4,
    )
    display(validation)
else:
    print('Skipped. Checked-in artifacts below are the reproducible chapter result.')

## Locked-test release

This cell refuses to run unless the saved three-run validation majority is at least 85%. It then evaluates the already frozen program on the test exactly once as a three-run confirmatory measurement.

In [ ]:
if RUN_PAID_GEPA:
    from chapter06.experiments.gepa_expanded.run_experiment import finalize_result
    test = evaluate_repeated(
        program_path=candidate_dir / 'optimized_program.json',
        split_name='test', output_dir=candidate_dir / 'test', repeats=3,
        validation_summary_path=candidate_dir / 'validation' / 'summary.json',
        minimum_validation_accuracy_pct=85.0, stage_cap_usd=2.0, threads=4,
    )
    reproduced_final = finalize_result(
        candidate_dir=candidate_dir, test_evaluation_dir=candidate_dir / 'test',
        final_dir=RESULTS / 'reproduction' / 'final',
    )
    display(test)
    display(reproduced_final)
else:
    print('Skipped locked-test execution.')

## Final comparison

Once the run is complete, this section reads only the frozen artifacts. Cost is split between optimizer compilation and the complete experiment ledger; latency is measured from fresh inference calls. The validation gate reached 88.3%, while the previously untouched test reached 76.25%. That shortfall from the aspirational 80% target is retained rather than tuning against the observed holdout.

In [ ]:
if (FINAL / 'summary.json').exists():
    final = json.loads((FINAL / 'summary.json').read_text())
    comparison = pd.DataFrame([
        {'program': 'Unoptimized Luna', 'test_accuracy_pct': final['baseline_test_accuracy_pct'],
         'correct': final['baseline_test_correct'], 'mean_latency_seconds': final['baseline_mean_latency_seconds']},
        {'program': 'GEPA-optimized Luna', 'test_accuracy_pct': final['optimized_test_accuracy_pct'],
         'correct': final['optimized_test_correct'], 'mean_latency_seconds': final['optimized_mean_latency_seconds']},
    ])
    display(comparison)
    display(pd.DataFrame([{
        'uplift_points': final['absolute_uplift_pct_points'],
        'relative_uplift_pct': final['relative_uplift_pct'],
        'optimizer_cost_usd': final['optimize_cost_usd'],
        'total_experiment_cost_usd': final['total_experiment_cost_usd'],
        'optimization_minutes': final['optimize_time_seconds'] / 60,
        'mcnemar_p': final['paired_mcnemar_p_value'],
        'bootstrap_95pct_ci': (final['paired_bootstrap_ci_low_pct_points'], final['paired_bootstrap_ci_high_pct_points']),
    }]))
else:
    print('Final GEPA artifacts have not been generated yet.')

## What GEPA learned

The exact learned instruction and reloadable DSPy program are saved separately. This makes the optimizer's result inspectable rather than reducing it to a score.

In [ ]:
prompt_path = FINAL / 'learned_instruction.txt'
print(prompt_path.read_text() if prompt_path.exists() else 'Learned prompt pending the paid GEPA run.')

## Reproducibility artifacts

The checked-in result directory contains the complete console log, task and reflection LM histories, GEPA candidate trace, reloadable program JSON, exact prompt, each uncached prediction, budget ledger, latency summaries, and paired statistical analysis. Histories are sanitized before persistence.